In [1]:
import pandas as pd
df = pd.read_csv("C:/Users/Ali Aghili/Desktop/superstore-analysis/data/raw/Sample - Superstore.csv", encoding='cp1252')


# پیدا کردن Product IDهایی که بیش از یک Product Name دارند
check = df.groupby('Product ID')['Product Name'].nunique()
mismatch_ids = sorted(check[check > 1].index.tolist())

rows = []
for pid in mismatch_ids:
    names = df[df['Product ID'] == pid]['Product Name'].unique()
    rows.append({
        'product_id': pid,
        'name_variant_1': names[0],
        'name_variant_2': names[1] if len(names) > 1 else None
    })

conflicts_df = pd.DataFrame(rows)
print(f"Total conflicting Product IDs: {len(conflicts_df)}")

conflicts_df.to_csv("C:/Users/Ali Aghili/Desktop/superstore-analysis/docs/data_quality/product_id_name_conflicts.csv", index=False)
print("Saved.")


Total conflicting Product IDs: 32
Saved.


In [3]:
#بررسی چولگی داده ها
print("Sales skewness:", df['Sales'].skew())
print("Profit skewness:", df['Profit'].skew())
print("Discount skewness:", df['Discount'].skew())

Sales skewness: 12.97275234181623
Profit skewness: 7.561431562468343
Discount skewness: 1.6842947474238648


In [5]:
# Discount is bounded (0 to ~0.8) and likely discrete-leveled,
# not continuous — checking frequency distribution instead of IQR
print(df['Discount'].value_counts().sort_index())

Discount
0.00    4798
0.10      94
0.15      52
0.20    3657
0.30     227
0.32      27
0.40     206
0.45      11
0.50      66
0.60     138
0.70     418
0.80     300
Name: count, dtype: int64


In [7]:
def iqr_bounds(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

# Merge order_items-equivalent columns with Category for per-category analysis
subset = df[['Category', 'Sales', 'Profit']]

for cat in subset['Category'].unique():
    cat_data = subset[subset['Category'] == cat]
    low_s, high_s = iqr_bounds(cat_data['Sales'])
    low_p, high_p = iqr_bounds(cat_data['Profit'])
    n_outliers_sales = ((cat_data['Sales'] < low_s) | (cat_data['Sales'] > high_s)).sum()
    n_outliers_profit = ((cat_data['Profit'] < low_p) | (cat_data['Profit'] > high_p)).sum()
    print(f"{cat}: Sales outliers={n_outliers_sales}/{len(cat_data)}, "
          f"Profit outliers={n_outliers_profit}/{len(cat_data)}")

Furniture: Sales outliers=164/2121, Profit outliers=407/2121
Office Supplies: Sales outliers=815/6026, Profit outliers=1053/6026
Technology: Sales outliers=172/1847, Profit outliers=250/1847


In [9]:
# Are Sales outliers (flagged by per-category IQR) simply large 
# legitimate bulk purchases, or something else?
def iqr_bounds(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

for cat in df['Category'].unique():
    cat_data = df[df['Category'] == cat]
    low, high = iqr_bounds(cat_data['Sales'])
    outliers = cat_data[cat_data['Sales'] > high]
    print(f"\n{cat}: {len(outliers)} high-Sales outliers")
    print(f"  Avg Quantity in outliers: {outliers['Quantity'].mean():.1f}")
    print(f"  Avg Quantity overall:     {cat_data['Quantity'].mean():.1f}")


Furniture: 164 high-Sales outliers
  Avg Quantity in outliers: 6.6
  Avg Quantity overall:     3.8

Office Supplies: 815 high-Sales outliers
  Avg Quantity in outliers: 4.9
  Avg Quantity overall:     3.8

Technology: 172 high-Sales outliers
  Avg Quantity in outliers: 5.6
  Avg Quantity overall:     3.8
